# 行业优先两层策略：收口复核

本笔记只复核紧凑收口指标和保留数据资产，不保留或重建已淘汰模型。

In [1]:
from pathlib import Path
import hashlib
import json
import sqlite3
import pandas as pd

root = Path.cwd()
while root.parent != root and not (root / 'PROJECT_STATUS.json').exists():
    root = root.parent
assert (root / 'PROJECT_STATUS.json').exists(), 'repository root not found'
closeout_path = root / 'docs/research/SECTOR_FIRST_TWO_STAGE_CLOSEOUT_2026-08-19.json'
closeout = json.loads(closeout_path.read_text(encoding='utf-8'))
closeout['decision']

{'production_change': False,
 'paper_tracking_change': False,
 'forward_comparison_change': False,
 'challenger_passed': False,
 'dynamic_v3_weight_test_run': False,
 'conclusion': 'do_not_use_a_sector_first_gate_or_fit_a_v3_switch',
 'reason': 'the sector layer had no stable forward rank signal and the nested challenger failed five of six preregistered gates'}

## 门禁与走步结果

独立重算通过门禁数量、正收益折数和区间是否跨零。

In [2]:
gates = closeout['candidate_results']['gates']
nested = closeout['candidate_results']['nested_walk_forward']
folds = nested['fold_mean_lift_bps']
checks = {
    'passed_gate_count': sum(bool(value) for value in gates.values()),
    'total_gate_count': len(gates),
    'positive_fold_count_recomputed': sum(value > 0 for value in folds),
    'outer_ci_crosses_zero': nested['ci95_bps'][0] < 0 < nested['ci95_bps'][1],
    'recent_mean_is_negative': closeout['candidate_results']['recent_diagnostic']['mean_lift_bps'] < 0,
}
assert checks == {
    'passed_gate_count': 1,
    'total_gate_count': 6,
    'positive_fold_count_recomputed': 1,
    'outer_ci_crosses_zero': True,
    'recent_mean_is_negative': True,
}
pd.Series(checks, name='recomputed')

passed_gate_count                    1
total_gate_count                     6
positive_fold_count_recomputed       1
outer_ci_crosses_zero             True
recent_mean_is_negative           True
Name: recomputed, dtype: object

## 信号分层诊断

行业层整体无排序能力，而个股行业相对残差保留弱正相关；这支持停止行业硬门槛，并只把后者登记为下一轮假设。

In [3]:
model = closeout['model_diagnostics']
rank_table = pd.DataFrame([
    {
        'layer': 'sector gate',
        'days': model['sector_layer']['available_days'],
        'mean_daily_rank_ic': model['sector_layer']['mean_daily_rank_ic'],
        'positive_day_share': model['sector_layer']['positive_rank_ic_share'],
        'top_minus_all_bps': model['sector_layer']['mean_top3_minus_all_sectors_bps'],
    },
    {
        'layer': 'stock sector-relative residual',
        'days': model['stock_sector_relative_residual_layer']['available_days'],
        'mean_daily_rank_ic': model['stock_sector_relative_residual_layer']['mean_daily_rank_ic'],
        'positive_day_share': model['stock_sector_relative_residual_layer']['positive_rank_ic_share'],
        'top_minus_all_bps': model['stock_sector_relative_residual_layer']['mean_top3_minus_all_candidates_bps'],
    },
])
assert rank_table.loc[0, 'mean_daily_rank_ic'] < 0
assert rank_table.loc[1, 'mean_daily_rank_ic'] > 0
rank_table

,layer,days,mean_daily_rank_ic,positive_day_share,top_minus_all_bps
0,sector gate,471,-0.009012,0.492569,-1.036646
1,stock sector-relative residual,467,0.074676,0.614561,33.449967


## 保留快照真实性

核对文件哈希、行业日行数、行业数和日期边界。

In [4]:
asset = closeout['retained_asset']
snapshot_path = root / asset['path']
actual_sha = hashlib.sha256(snapshot_path.read_bytes()).hexdigest()
with sqlite3.connect(f'file:{snapshot_path}?mode=ro&immutable=1', uri=True) as connection:
    row = connection.execute(
        'SELECT COUNT(*), COUNT(DISTINCT sector_code), MIN(trade_day), MAX(trade_day) FROM sector_state'
    ).fetchone()
snapshot_check = {
    'sha256_match': actual_sha == asset['sha256'],
    'row_count': row[0],
    'sector_count': row[1],
    'observed_from': row[2],
    'observed_to': row[3],
}
assert snapshot_check == {
    'sha256_match': True,
    'row_count': asset['rows'],
    'sector_count': asset['sectors'],
    'observed_from': asset['observed_from'],
    'observed_to': asset['observed_to'],
}
pd.Series(snapshot_check, name='snapshot')

sha256_match           True
row_count             20150
sector_count             31
observed_from    2023-10-20
observed_to      2026-06-26
Name: snapshot, dtype: object

## 复核结论

紧凑结论内部一致，保留快照与登记哈希完全一致。失败模型未重建；生产、纸面和 live-facing 行为均不应改变。